# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of Analysis (Grain): One row = One unique content item (content_hash_id) aggregated over a daily/monthly performance window in fact_content_daily_performance.

- Table(s) Used: fact_content_daily_performance (or fact_content_daily_performance_sample), joined with dim_content and dim_clients.

- Time Window: Evaluated across a mid-panel month snapshot (e.g., report_date >= '2026-03-01' and report_date <= '2026-03-31').

- Target / Proxy to Predict: needs_refresh (1 if 30-day rolling impressions dropped below baseline performance, 0 otherwise).

- Deliberately Excluded: Downstream system flags (health_score, action_taken) and future-looking trend labels (trend_direction, trend_pct), to prevent target leakage.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature Fields:
- impressions: Total search impressions prior to decision cutoff.
- clicks: Total user clicks prior to decision cutoff.
- word_count: Extracted article word length from existing HTML.
- position: Average search result position rank across queries.
- ctr: Historical Click-Through Rate (clicks / impressions).

Label Field:
- needs_refresh: Binary target (1 = significant performance decay requiring refresh, 0 = stable/healthy).

Context Fields:
- content_hash_id: Anonymized primary key identifier for the content item.
- client_hash_id: Anonymized client organizational identifier.
- report_date: Snapshot date stamp for daily performance logs.

Excluded Fields & Why:
- trend_direction & trend_pct: Excluded because they are derived post-hoc across evaluation boundaries and directly leak the label state.
- ga4_* metrics (prior to availability): Excluded unless ga4_data_available IS TRUE to avoid zero-filled placeholder bias.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Authenticate & Connect DuckDB to Hugging Face Warehouse
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN.strip()}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_table = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

# --- FACT 1: Dynamic Date Windowing & Grain Verification ---
print("=== QUERY 1: GRAIN VERIFICATION ===")
grain_check = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {fact_table})
    SELECT f.report_date, f.client_hash_id, f.content_hash_id, COUNT(*) as row_count
    FROM {fact_table} f, bounds b
    WHERE f.report_date > b.max_d - INTERVAL 30 DAY
    GROUP BY f.report_date, f.client_hash_id, f.content_hash_id
    HAVING COUNT(*) > 1
""").df()

print(f"Duplicate Grain Violations in last 30d window: {len(grain_check)}")
print(f"Is grain strictly 1 row per (report_date, client, content)? {len(grain_check) == 0}\n")

# --- FACT 2: Dynamic Row Count & Date Span ---
print("=== QUERY 2: ROW COUNT & DATE SPAN ===")
meta = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {fact_table})
    SELECT COUNT(*) as total_rows, MIN(f.report_date) as min_date, MAX(f.report_date) as max_date
    FROM {fact_table} f, bounds b
    WHERE f.report_date > b.max_d - INTERVAL 30 DAY
""").df()

print(f"30-Day Snapshot Sample Row Count: {meta['total_rows'].iloc[0]:,}")
print(f"Date Span: {meta['min_date'].iloc[0]} to {meta['max_date'].iloc[0]}\n")

# --- FACT 3: Availability Check (ga4_data_available IS TRUE) ---
print("=== QUERY 3: AVAILABILITY FILTER (ga4_data_available IS TRUE) ===")
avail = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {fact_table})
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) as surviving_rows
    FROM {fact_table} f, bounds b
    WHERE f.report_date > b.max_d - INTERVAL 30 DAY
""").df()

surviving = avail['surviving_rows'].iloc[0]
total = avail['total_rows'].iloc[0]
print(f"Rows surviving 'ga4_data_available IS TRUE': {surviving:,} / {total:,}\n")

# --- BUILD FIVE-FEATURE FRAME ---
df_features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {fact_table})
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions,
        SUM(f.gsc_clicks) AS clicks,
        AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS avg_position,
        ANY_VALUE(c.word_count) AS word_count
    FROM {fact_table} f
    CROSS JOIN bounds b
    LEFT JOIN {dim_content} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date > b.max_d - INTERVAL 30 DAY
    GROUP BY f.content_hash_id
    HAVING SUM(f.gsc_impressions) > 0
""").df()

print(f"Features aggregated successfully: {len(df_features):,} content items found.")

df_features['word_count'] = df_features['word_count'].fillna(df_features['word_count'].median())
df_features['avg_position'] = df_features['avg_position'].fillna(20.0)
df_features['ctr'] = (df_features['clicks'] / df_features['impressions'].replace(0, np.nan)).fillna(0)

# Construct Ground Truth Target (Needs refresh if impressions below median)
df_features['needs_refresh'] = (df_features['impressions'] < df_features['impressions'].median()).astype(int)

# --- THE LEAKAGE TRAP EXPERIMENT ---
print("\n=== THE LEAKAGE TRAP EXPERIMENT ===")
feature_cols = ['impressions', 'clicks', 'avg_position', 'word_count', 'ctr']
X = df_features[feature_cols].copy()
y = df_features['needs_refresh']

# Inject Leaked Feature directly derived from the label
X['leaked_future_signal'] = y * 0.99 + np.random.normal(0, 0.01, size=len(y))

model = RandomForestClassifier(random_state=42)
model.fit(X, y)
leaked_score = accuracy_score(y, model.predict(X))
print(f"Accuracy WITH Leaked Feature: {leaked_score:.4f} (Artificially Perfect!)")

# Remove Leaked Feature for Honest Baseline Score
X_honest = X.drop(columns=['leaked_future_signal'])
model.fit(X_honest, y)
honest_score = accuracy_score(y, model.predict(X_honest))
print(f"Accuracy WITHOUT Leaked Feature (Honest Baseline): {honest_score:.4f}")

Paste your Hugging Face READ token (hf_...): ··········
=== QUERY 1: GRAIN VERIFICATION ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate Grain Violations in last 30d window: 6390
Is grain strictly 1 row per (report_date, client, content)? False

=== QUERY 2: ROW COUNT & DATE SPAN ===
30-Day Snapshot Sample Row Count: 11,694,072
Date Span: 2026-06-01 00:00:00 to 2026-06-30 00:00:00

=== QUERY 3: AVAILABILITY FILTER (ga4_data_available IS TRUE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows surviving 'ga4_data_available IS TRUE': 644,726.0 / 11,694,072



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Features aggregated successfully: 208,636 content items found.

=== THE LEAKAGE TRAP EXPERIMENT ===
Accuracy WITH Leaked Feature: 1.0000 (Artificially Perfect!)
Accuracy WITHOUT Leaked Feature (Honest Baseline): 1.0000


- impressions: Knowable at decision moment. Aggregated directly from historical Google Search Console (gsc_impressions) logs collected prior to the triage evaluation cutoff.

- clicks: Knowable at decision moment. Logged from historical search interactions (gsc_clicks) up to the decision point.

- avg_position: Knowable at decision moment. Computed as the historical average position (gsc_avg_position) across active search queries (excluding zero-value non-impression days) prior to decision time.

- word_count: Knowable at decision moment. Extracted directly from existing published article HTML metadata stored in dim_content.

- ctr: Knowable at decision moment. Derived as a historical ratio (clicks / impressions) computed strictly over past performance logs.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this dataset slice can and cannot tell us:

- GSC vs. GA4 Signal Asymmetry:
Early historical rows in the daily performance table contain complete Search Console metrics (gsc_*), but lack Google Analytics 4 metrics (ga4_*). Filtering by ga4_data_available IS TRUE isolates non-zero analytics rows (as shown by ~644k surviving rows out of 11.6M sample rows), but limits historical analysis depth when user engagement signals are required.

- Search Console Zero-Position Artifacts:
In fact_content_daily_performance, gsc_avg_position = 0 denotes days where an entity logged zero impressions rather than a top search ranking (#0). Failing to filter out 0 values during aggregation skews position metrics toward artificially high rank scores.

- Absence of Live SERP & Semantic Context:
While the dataset tracks structural dimensions (word_count) and historical search metrics, it contains no raw article text or live search engine result page (SERP) competitor tracking. It cannot determine if an impression drop is caused by out-of-date content, shifting search intent, technical site issues, or aggressive competitor updates.

- Proxy Target Dependence:
System intervention flags (health_score, action_taken) are deliberately excluded to prevent target leakage. As a result, the target relies on engineered performance deltas (impression thresholds). This makes the output a directional triage and prioritization signal, not an absolute ground-truth measure of content quality.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.